In [1]:
import sys
sys.path.append("scripts/") 

In [2]:
import os
import sys
import numpy as np
from time import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [3]:
def cov_matrix_vectorized(x):
    # x should be of shape (timesteps, features) or (features, timesteps)
    if x.shape[0] < x.shape[1]:
        x = x.T
    cov = np.cov(x, rowvar=False)
    return cov.flatten()

In [4]:
ML_DATA_PATH = "/project/scratch/p200631/Silvana/gpu_utilization/results/60-random/ml.npz"

In [5]:
ml_data = np.load(ML_DATA_PATH)
X_train, y_train, X_test, y_test = ml_data['X_train'], ml_data['y_train'],ml_data['X_test'],ml_data['y_test']


In [6]:
X_train = X_train.reshape(X_train.shape[0],-1)
X_test = X_test.reshape(X_test.shape[0],-1)

In [7]:
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

(14162, 3780) (14162,)
(3541, 3780) (3541,)


In [8]:
pipeline_arg_list = [
        ('clf', SVC()), 
]

In [9]:
pipeline_arg_list.insert(0,('scaler', StandardScaler()),)
pipeline_arg_list.insert(1,('pca', PCA()),)

In [10]:
pipeline = Pipeline(pipeline_arg_list)


In [11]:
parameters_list = [
    {
        'clf':(SVC(),),
        'clf__C':(0.1,1.0,10.0),
        'clf__kernel':('linear',),
    },
    {
        'clf':(RandomForestClassifier(),),
        'clf__n_estimators':(50,100,250),
    },
]

In [12]:
for params in parameters_list:
        params['pca__n_components'] = (28,64,256,512,)

In [13]:
for parameters in parameters_list:

    print('****************************************')
    print('Running {}'.format(parameters['clf'][0]))
    print('****************************************')

    grid_search = GridSearchCV(pipeline,
                               parameters,
                               n_jobs=-1,
                               verbose=1,
                               cv=10)

    print('Performing grid search...\n')
    print('Pipeline: {}\n'.format([name for name, _ in pipeline.steps]))
    print('Pipeline parameters:\n')
    for k,v in parameters.items():
        print('  {}:{}'.format(k,v))
    print('\r')
    t0 = time()

    # Fit the GridSearch on the training data
    grid_search.fit(X_train,y_train)
    print('Done in {:0.3f}s\n'.format(time()-t0))
          
    print('Best training score: {:0.4f}\n'.format(grid_search.best_score_))
    best_model_str = [ str(tup[1])[:-2] for tup in grid_search.best_estimator_.__dict__['steps'] if tup[0]=='clf' ][0]
    best_model_params = [params for params in parameters_list if best_model_str==str(params['clf'][0])[:-2]][0]
    best_params = grid_search.best_estimator_.get_params()
    print('Best parameters:\n')
    for param_name in sorted(best_model_params.keys()):
        print('  {}: {}'.format(param_name, best_params[param_name]))
    print('\nTEST SET ACCURACY USING BEST HYPERPARAMETERS {:0.4f}\n'.format(grid_search.score(X_test,y_test)))


****************************************
Running SVC()
****************************************
Performing grid search...

Pipeline: ['scaler', 'pca', 'clf']

Pipeline parameters:

  clf:(SVC(),)
  clf__C:(0.1, 1.0, 10.0)
  clf__kernel:('linear',)
  pca__n_components:(28, 64, 256, 512)

Fitting 10 folds for each of 12 candidates, totalling 120 fits
Done in 834.044s

Best training score: 0.7534



IndexError: list index out of range